# BPS Djuren — Least-Squares Direct Fit (Debug)

Replicates djuren-test.ipynb exactly but using BPS_fast's blend weights + onering coords.

Key fix vs previous attempt: **no constant term** in the polynomial basis, matching
Djuren.py's `include_constant=False`. This ensures the vertex function passes through
the central vertex (poly(0,0) = 0), keeps the system overdetermined for valence-6
vertices (6 eqs, 5 unknowns for degree 2), and avoids ill-conditioned solutions.

In [1]:
import numpy as np
import torch
import json, importlib
import open3d as o3d
o3d.utility.set_verbosity_level(o3d.utility.VerbosityLevel.Error)

import BPS; importlib.reload(BPS)
from BPS import BPS_fast

Using open3d version at ['/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/open3d']
Using open3d version at ['/Users/romywilliamson/miniconda3/envs/bens/lib/python3.8/site-packages/open3d']


In [2]:
config_filepath = "configs/surfaces/fertility500-djuren.json"
with open(config_filepath) as f:
    config_dict = json.load(f)

surface_config = config_dict['surface-config']
name = surface_config['coarse_patches_id']

device = 'cpu'
bps = BPS_fast(surface_config, device=device)
print(f"{name}: {bps.n} verts, {bps.F.shape[0]} faces, degree {bps.degree}")

Initialising BPS on cpu
HalfEdgeTriangleMesh with 244 points and 1500 half edges.
mean edge length tensor(0.1616)
Using Local Scales
Assiging a halfedge to each vertex.
Preparing V_he for halfedge mesh
Prepared V_he for halfedge mesh.

Optimising V_he.
Finished optimising V_he.

base triangle verts torch.float32
sharp edges obj []
sharp edges o3d []
sharp halfedges: []
Computing rotations.
fertility500: 244 verts, 500 faces, degree 2


In [3]:
# ── poly basis WITHOUT constant term — matches Djuren.py include_constant=False ──
# poly(0,0) = 0, so vertex function naturally passes through V[v] at the origin.

def poly_basis_nc(u, v, degree, stack_dim=-1):
    """Polynomial basis starting at degree 1 (no constant term).
    stack_dim=-1 → (..., num_terms)  [for 1-D fitting inputs]
    stack_dim=1  → (F, num_terms, S, 3)  [for 3-D evaluation inputs]
    """
    terms = []
    for d in range(1, degree + 1):
        for i in range(d, -1, -1):
            j = d - i
            terms.append((u ** i) * (v ** j))
    return torch.stack(terms, dim=stack_dim)

degree    = bps.degree
num_terms = int((degree + 1) * (degree + 2) / 2) - 1   # no constant
print(f"degree={degree}, num_terms (no const)={num_terms}")

degree=2, num_terms (no const)=5


In [4]:
# ── precompute blend weights and (normalised) onering coords ──────────────────
mesh_res   = 6
base_patch = o3d.io.read_triangle_mesh(f'data/high_precision_subdiv_triangles/triangle_{mesh_res}.obj')
x_np       = np.asarray(base_patch.vertices)[:, :2]   # (S, 2)
base_faces = np.asarray(base_patch.triangles)          # (T, 3)
S          = x_np.shape[0]
F_count    = bps.F.shape[0]

x_t = torch.tensor(x_np, dtype=torch.float32).unsqueeze(0).expand(F_count, -1, -1).clone()

precomp        = bps.precompute_data_from_samples(x_t, detached=True)
blend_weights  = precomp['blend_weights']    # (F, S, 3)
onering_coords = precomp['onering_coords']   # (F, S, 3, 2)  — normalised by local_scale

# Scale back to raw (3-D metric) one-ring units
# bps.local_scales[f, p] = mean edge length at vertex F[f][p]
ls     = bps.local_scales                                       # (F, 3)
raw_oc = onering_coords * ls.unsqueeze(1).unsqueeze(-1)         # (F, S, 3, 2)

print('blend_weights:', blend_weights.shape)
print('raw_oc:',        raw_oc.shape)

Precomputing blend weights etc, using djuren blending.


/Users/romywilliamson/Documents/BNS/BCSv2/BPS.py:976: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  V_face = torch.tensor(self.V[verts, :], dtype=torch.float32, device=x.device)   # (3,3)
/Users/romywilliamson/Documents/BNS/BCSv2/bns_utils.py:350: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  v_i = torch.tensor(V_face[i, :], dtype=torch.float32)
/Users/romywilliamson/Documents/BNS/BCSv2/bns_utils.py:351: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  v_ip1 = torch.tensor(V_face[(i+1)%3, :], dtype=torch.float32)
/Users/romywilliamson/Docume

Now deadling with poly basis
blend_weights: torch.Size([500, 2145, 3])
raw_oc: torch.Size([500, 2145, 3, 2])


In [5]:
# ── fit W[v] Djuren-style for every vertex ────────────────────────────────────
two_pi = 2 * torch.pi
W_all  = torch.zeros(bps.n, num_terms, 3)

for v in range(bps.n):
    onering  = bps.onerings[v]
    valence  = onering['valence']
    cum_ang  = onering['cumulative_angles']

    if cum_ang is None:
        continue   # boundary — W stays 0

    total     = cum_ang[-1]
    V_indices = onering['V_indices'][:valence]
    Vv        = bps.V[v].cpu()

    # Flat-djuren angles: V_indices[0] at 0, V_indices[j≥1] at cum_ang[j-1]*2π/total
    flat_angles = torch.zeros(valence)
    for j in range(1, valence):
        flat_angles[j] = cum_ang[j - 1] * two_pi / total

    # Raw 3-D edge lengths to each neighbour
    r_raw = torch.tensor([
        torch.norm(bps.V[V_indices[j]].cpu() - Vv).item()
        for j in range(valence)
    ])

    # One-ring coords of neighbours (raw metric)
    ox = r_raw * torch.cos(flat_angles)
    oy = r_raw * torch.sin(flat_angles)

    # Poly basis — NO constant term: (valence, num_terms)
    P = poly_basis_nc(ox, oy, degree, stack_dim=-1)

    # Targets: V[neighbour] - V[v]  (3-D displacement)
    Y = torch.stack([
        (bps.V[V_indices[j]].cpu() - Vv).float()
        for j in range(valence)
    ])   # (valence, 3)

    # Least-squares (overdetermined for valence > num_terms, as desired)
    W_all[v] = torch.linalg.lstsq(P, Y).solution

print('Fitting done.  W_all:', W_all.shape)
# Sanity: at origin poly=0, so vertex func = V[v].  Check by verifying W has no blow-ups.
norms = W_all.norm(dim=-1).max(dim=-1).values
print(f'Max coefficient norm per vertex: max={norms.max():.4f}, median={norms.median():.4f}')

Fitting done.  W_all: torch.Size([244, 5, 3])
Max coefficient norm per vertex: max=19.8834, median=4.2347


In [6]:
# ── evaluate vertex functions at all sample points ────────────────────────────
# poly_basis_nc on (F, S, 3) inputs → (F, num_terms, S, 3)
X_raw  = raw_oc[:, :, :, 0]   # (F, S, 3)
Y_raw  = raw_oc[:, :, :, 1]   # (F, S, 3)
pb_raw = poly_basis_nc(X_raw, Y_raw, degree, stack_dim=1)  # (F, num_terms, S, 3)

# W_hat[f, p, :, :] = W_all[F[f][p]]  →  (F, 3, num_terms, 3)
F_idx  = torch.tensor(bps.F, dtype=torch.long)
W_hat  = W_all[F_idx]   # (F, 3, num_terms, 3)

# vertex function values (displacement part): (F, 3, S, 3)
poly_vals = torch.einsum('fbsp,fpbc->fpsc', pb_raw, W_hat)

# Add vertex positions
per_face_verts = bps.V[F_idx].unsqueeze(2)   # (F, 3, 1, 3)
vertex_funcs   = poly_vals + per_face_verts    # (F, 3, S, 3)

# ── blend ─────────────────────────────────────────────────────────────────────
output = torch.einsum('fsp,fpsc->fsc', blend_weights, vertex_funcs)   # (F, S, 3)
print('output:', output.shape, '  nan?', output.isnan().any().item())

output: torch.Size([500, 2145, 3])   nan? False


In [7]:
# ── topological weld + visualise ──────────────────────────────────────────────
output_np = output.detach().numpy()

bary_raw = np.stack([
    bps.bary_weight(torch.tensor(x_np, dtype=torch.float32), k).numpy()
    for k in range(3)
], axis=-1)   # (S, 3)

R_sub    = int(np.round((-3 + np.sqrt(9 + 8 * (S - 1))) / 2))
int_bary = np.round(bary_raw * R_sub).astype(int)

hash_to_id = {}
global_V, global_F = [], []

for f_i in range(F_count):
    cv  = [int(bps.F[f_i, k]) for k in range(3)]
    Vp  = output_np[f_i]
    l2g = {}
    for s in range(S):
        u, v, w = int_bary[s]
        v0, v1, v2 = cv
        if   u == R_sub: sig = f"V_{v0}"
        elif v == R_sub: sig = f"V_{v1}"
        elif w == R_sub: sig = f"V_{v2}"
        elif u == 0:
            mn,mx = min(v1,v2),max(v1,v2); d = v if mn==v1 else w
            sig = f"E_{mn}_{mx}_{d}"
        elif v == 0:
            mn,mx = min(v0,v2),max(v0,v2); d = u if mn==v0 else w
            sig = f"E_{mn}_{mx}_{d}"
        elif w == 0:
            mn,mx = min(v0,v1),max(v0,v1); d = u if mn==v0 else v
            sig = f"E_{mn}_{mx}_{d}"
        else:
            sig = f"F_{f_i}_{u}_{v}_{w}"
        if sig not in hash_to_id:
            hash_to_id[sig] = len(global_V)
            global_V.append(Vp[s])
        l2g[s] = hash_to_id[sig]
    for tri in base_faces:
        global_F.append([l2g[tri[0]], l2g[tri[1]], l2g[tri[2]]])

mesh = o3d.geometry.TriangleMesh()
mesh.vertices  = o3d.utility.Vector3dVector(np.array(global_V))
mesh.triangles = o3d.utility.Vector3iVector(np.array(global_F))
mesh.compute_vertex_normals()

out_path = f"data/surfaces/{name}-bps-lstsq-debug.obj"
o3d.io.write_triangle_mesh(out_path, mesh)
print(f"Saved {out_path}")
o3d.visualization.draw_geometries([mesh], window_name='BPS lstsq debug — welded')

Saved data/surfaces/fertility500-bps-lstsq-debug.obj
